# Forensic Analyst — Agentic RAG System
Upload documents (PDF, XLSX, images, etc.), build a knowledge graph, and chat with an intelligent agent that can search, analyze, and run compliance checks.

In [ ]:
import os
import sys
import asyncio
import nest_asyncio
from pathlib import Path
from dotenv import load_dotenv

nest_asyncio.apply()
load_dotenv()

# Ensure project root is on sys.path
PROJECT_ROOT = str(Path.cwd())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Agno's Gemini model reads GOOGLE_API_KEY
if os.getenv("GEMINI_API_KEY") and not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"]

print("Environment ready")

In [ ]:
from skills.shared.lightrag_init import get_rag_instance

# Initialize LightRAG (Mistral for KG construction, local embeddings)
rag = await get_rag_instance()
print("LightRAG initialized")

## Document Indexing (Optional)
Place your input documents in `user_documents/` and use the agent's `upload_document` tool for smart preprocessing, or run the cell below to batch-index files directly.

In [ ]:
from skills.shared.preprocessors import PreprocessingRouter
from skills.shared.lightrag_init import DEFAULT_INPUT_DIR

router = PreprocessingRouter()
INPUT_DIR = DEFAULT_INPUT_DIR

async def process_and_index(file_paths: list[str]):
    for path in file_paths:
        print(f"Processing {path}...")
        result = await router.process(path)
        text = result.get("text", "")
        if not text.strip():
            print(f"  Skipped (no text extracted)")
            continue
        print(f"  Extracted {len(text):,} chars, indexing...")
        await rag.ainsert(text)
        print(f"  Done: {Path(path).name}")

# --- Add filenames from user_documents/ ---
files_to_index = [
    # str(INPUT_DIR / 'sample_docs/sample_statement.pdf'),
    # str(INPUT_DIR / 'financial_report.xlsx'),
]

if files_to_index:
    await process_and_index(files_to_index)
else:
    print(f"No files to index. Place documents in {INPUT_DIR} and add paths above.")

## Agent Chat
The forensic analyst agent has 9 tools: search, upload, delete, list docs, compliance check (full or per-phase), report, preview, and Excel reader.

In [ ]:
from agno.agent import Agent
from skills.shared.agent_tools import ForensicToolkit
from skills.shared.model_config_agno import get_agent_model

toolkit = ForensicToolkit(rag=rag)

agent = Agent(
    name="Forensic Analyst",
    model=get_agent_model(),
    tools=[toolkit],
    instructions=[
        "You are a forensic financial analyst with access to document analysis and compliance tools.",
        "",
        "## Search Strategy",
        "- Always use search_knowledge_graph before answering factual questions.",
        "- If initial results are insufficient, reformulate your query or try a different mode:",
        "  - 'local' for specific entities/facts, 'global' for themes/summaries, 'hybrid' for balanced.",
        "- Decompose complex questions into sub-queries and search iteratively.",
        "",
        "## Document Handling",
        "- Use preview_document to inspect extraction quality before indexing.",
        "- For Excel files, use read_excel_sheet for targeted analysis without full indexing.",
        "- Use upload_document to index new documents (auto-selects best preprocessor).",
        "",
        "## Compliance Workflow",
        "- Use run_compliance_check for a full 5-phase pipeline (profile → atomise → retrieve → adjudicate → report).",
        "- Use run_compliance_phase to run individual phases if debugging.",
        "- Use get_compliance_report to retrieve the latest report.",
        "",
        "## Response Style",
        "- Ground answers in search results. If documents don't contain the answer, say so.",
        "- When citing numbers, include the source context.",
        "- Explain your tool choices and reasoning when performing multi-step analysis.",
    ],
    markdown=True,
)
print(f"Agent ready — model: {agent.model.id}, tools: {[t.name for t in agent.tools]}")

In [ ]:
print("Chat started. Type 'quit' or 'exit' to stop.\n")

try:
    while True:
        user_input = input("You: ")
        if user_input.strip().lower() in ("quit", "exit", "q"):
            print("Goodbye!")
            break
        if not user_input.strip():
            continue
        response = await agent.arun(user_input)
        print(f"\nAssistant: {response.content}\n")
except KeyboardInterrupt:
    print("\nChat stopped.")

## One-Shot Compliance (Optional)
Run the cell below to trigger a full compliance check via the agent.

In [ ]:
# One-shot compliance check via agent
response = await agent.arun("Run a full compliance check on the indexed documents and summarize the results.")
print(response.content)